# Vector RAG vs Graph RAG — Evaluation on Zync Documentation

This notebook evaluates two retrieval strategies on five documents about **Zync**, a fictional code collaboration platform with its own made-up CLI. The commands (`zync upload`, `zync lane rebase`, `zync auth refresh`, etc.) do not exist in any real tool — the LLM has no pre-trained knowledge about them.

- **Vector RAG**: embed each document, retrieve by cosine similarity, pass top results to an LLM.
- **Graph RAG**: extract entities and relationships from prose, store them in Neo4j, traverse the graph with Cypher, pass structured facts to an LLM.

Both methods run on the same five-document corpus and are measured on the same benchmark questions. The benchmark is designed around questions that require combining facts from multiple documents — a useful stress test for both approaches.

Ollama is used locally for embeddings and answer generation. Neo4j is used for graph storage and traversal.

## 1. The corpus — five interconnected company documents

Enterprise knowledge bases typically contain documents like:

- onboarding handbooks with product architecture notes mixed into HR guidance
- policy documents where supplier dependencies sit alongside compliance rules
- incident retrospectives that interleave business impact, operations, and finance commentary

What makes this corpus interesting for evaluation is that all five documents share a common set of entities — products, teams, suppliers, locations, and events — and many useful questions require combining facts from two or more of them.

The benchmark questions are designed to highlight this cross-document dependency. Both retrieval methods are evaluated on exactly the same questions, so the comparison is apples-to-apples.

## 2. Architecture overview

**Document-level vector baseline** — one embedding per full document, cosine similarity to retrieve top-k:

```
Query ──embed──► Query Vector
                     │
                     ▼  cosine similarity
  Doc-1 Vector ───── score
  Doc-2 Vector ───── score   ──► top-k full docs ──► LLM ──► Answer
  Doc-3 Vector ───── score
  Doc-4 Vector ───── score
  Doc-5 Vector ───── score
```

**Graph-oriented retrieval** — entities and relationships stored in Neo4j, traversed with Cypher:

```
Query ──tokenize──► Seed Entities ──Cypher hop──► Neighbour Entities
                                                        │
                                              (subject, relation, object) triples
                                                        │
                                                        ▼
                                             Context Builder ──► LLM ──► Answer
```

The key architectural difference: vector retrieval answers "which document is most similar to this query?", while graph retrieval answers "what entities and relationships are directly connected to the concepts in this query?" Each has strengths depending on the question type.

## 3. Local setup

Run these commands before executing the notebook:

```bash
pip install -r requirements.txt
ollama pull llama3.1:8b
ollama pull nomic-embed-text
docker run --rm --name vector-graph-rag-neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:5
ollama serve
```

Set these environment variables if you are not using the defaults:

```bash
export NEO4J_URI=bolt://localhost:7687
export NEO4J_USERNAME=neo4j
export NEO4J_PASSWORD=password
export NEO4J_DATABASE=neo4j
```

This notebook assumes Ollama is available at `http://localhost:11434` and Neo4j is reachable over Bolt.


In [2]:
from pathlib import Path
import os
import re
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from neo4j import GraphDatabase

pd.set_option("display.max_colwidth", 140)
plt.style.use("ggplot")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OLLAMA_BASE_URL = "http://localhost:11434"
GENERATION_MODEL = "gpt-oss:20b"
EMBEDDING_MODEL = "nomic-embed-text:latest"
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Expected a data directory at {DATA_DIR}. Open the notebook from the vector_vs_graph_rag folder."
    )

print(f"Notebook directory : {BASE_DIR}")
print(f"Corpus directory   : {DATA_DIR}")
print(f"Generation model   : {GENERATION_MODEL}")
print(f"Neo4j endpoint     : {NEO4J_URI}")

Notebook directory : /Users/aayushostwal/Desktop/aayush/ai-tutorials/vector_vs_graph_rag
Corpus directory   : /Users/aayushostwal/Desktop/aayush/ai-tutorials/vector_vs_graph_rag/data
Generation model   : gpt-oss:20b
Neo4j endpoint     : bolt://localhost:7687


In [4]:
def ollama_healthcheck(base_url: str = OLLAMA_BASE_URL) -> dict:
    """Confirm that the local Ollama server is reachable and models are present."""
    response = requests.get(f"{base_url}/api/tags", timeout=30)
    response.raise_for_status()
    payload = response.json()
    model_names = [item["name"] for item in payload.get("models", [])]
    return {
        "available_models": model_names,
        "embedding_model_present": EMBEDDING_MODEL in model_names,
        "generation_model_present": GENERATION_MODEL in model_names,
    }


def build_neo4j_driver(
    uri: str = NEO4J_URI,
    username: str = NEO4J_USERNAME,
    password: str = NEO4J_PASSWORD,
):
    """Create a Neo4j driver for graph ingestion and retrieval."""
    return GraphDatabase.driver(uri, auth=(username, password))


graph_driver = build_neo4j_driver()


def neo4j_healthcheck(driver=graph_driver, database: str = NEO4J_DATABASE) -> dict:
    """Confirm that Neo4j is reachable before loading the knowledge graph."""
    driver.verify_connectivity()
    with driver.session(database=database) as session:
        entity_count = session.run("MATCH (n:Entity) RETURN count(n) AS count").single()["count"]
        relationship_count = session.run("MATCH ()-[r:RELATES_TO]->() RETURN count(r) AS count").single()["count"]
    return {
        "uri": NEO4J_URI,
        "database": database,
        "entity_count": entity_count,
        "relationship_count": relationship_count,
    }


def ollama_embed(text: str, model: str = EMBEDDING_MODEL, base_url: str = OLLAMA_BASE_URL) -> np.ndarray:
    """Create one embedding vector using the local Ollama embeddings endpoint."""
    response = requests.post(
        f"{base_url}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=120,
    )
    response.raise_for_status()
    return np.array(response.json()["embedding"], dtype=float)


def ollama_generate(prompt: str, model: str = GENERATION_MODEL, base_url: str = OLLAMA_BASE_URL) -> str:
    """Generate a grounded answer through the local Ollama generation endpoint."""
    response = requests.post(
        f"{base_url}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=180,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


health = {
    "ollama": ollama_healthcheck(),
    "neo4j": neo4j_healthcheck(),
}
health


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Entity` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:Entity) RETURN count(n) AS count'
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `RELATES_TO` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classificati

{'ollama': {'available_models': ['nomic-embed-text:latest', 'gpt-oss:20b'],
  'embedding_model_present': True,
  'generation_model_present': True},
 'neo4j': {'uri': 'bolt://localhost:7687',
  'database': 'neo4j',
  'entity_count': 0,
  'relationship_count': 0}}

## 4. Load and explore the corpus

All five documents describe different aspects of **Zync**, a fictional code collaboration platform with its own CLI. The CLI commands (`zync upload`, `zync lane rebase`, `zync auth refresh`, etc.) are fabricated — they do not exist in any real tool, so the LLM has no pre-trained knowledge about them.

| Document | What it covers |
|---|---|
| **Tech Onboarding** | Core concepts (vaults, lanes, patches), the basic `zync` workflow, and how `zync upload` feeds into the Patch Store |
| **Design Documentation** | The four platform components (Zync Core, Patch Store, Lane Tracker, Proposal Engine, Guardian Service), their data flow, and which team owns each |
| **Contribution Guidelines** | How to raise a proposal with `zync proposal create`, the Guardian review and approval process, and the Vault Admin's role |
| **Common Errors and ZE Codes** | The four ZE-series error codes (ZE-401, ZE-302, ZE-404, ZE-500), what causes each, and the exact `zync` command that resolves it |
| **Access Control and Roles** | The three role levels (Contributor, Guardian, Vault Admin), what each can do, and which team owns the Guardian Service |

In [5]:
def parse_document(path: Path) -> dict:
    """Parse each text file into a title and body."""
    raw_text = path.read_text(encoding="utf-8")
    lines = [line.rstrip() for line in raw_text.splitlines()]
    title = path.stem
    body_lines = []

    for line in lines:
        if line.startswith("Title:"):
            title = line.replace("Title:", "", 1).strip()
            continue
        if line.strip():
            body_lines.append(line.strip())

    body = "\n".join(body_lines)
    return {
        "doc_id": path.name,
        "title": title,
        "body": body,
        "full_text": f"{title}\n\n{body}",
    }


documents = [parse_document(path) for path in sorted(DATA_DIR.glob("*.txt"))]
docs_df = pd.DataFrame(documents)
docs_df[["doc_id", "title"]]

,doc_id,title
0,01_tech_onboarding.txt,Tech Onboarding
1,02_design_documentation.txt,Design Documentation
2,03_contribution_guidelines.txt,Contribution Guidelines
3,04_common_errors.txt,Common Errors and ZE Codes
4,05_access_and_roles.txt,Access Control and Roles


## 5. Build the document-level vector baseline

The vector baseline uses one embedding per full document and retrieves by cosine similarity. This is a common first implementation and works well for questions that map cleanly to a single source document.

Design choices for this baseline:
- one embedding per full document (no chunking)
- no metadata filters
- `top_k=1` for benchmark evaluation

This approach is fast and simple to implement. The benchmark section will show how it handles questions that require facts from more than one document.

In [6]:
vector_index = []

for document in documents:
    vector_index.append({
        "doc_id": document["doc_id"],
        "title": document["title"],
        "text": document["full_text"],
        "embedding": ollama_embed(document["full_text"]),
    })

print(f"Built embeddings for {len(vector_index)} documents.")


def vector_retrieve(query: str, top_k: int = 5) -> list[dict]:
    """Retrieve semantically similar full documents using a document-level strategy."""
    query_embedding = ollama_embed(query)
    scored = []
    for item in vector_index:
        scored.append({
            "doc_id": item["doc_id"],
            "title": item["title"],
            "text": item["text"],
            "score": cosine_similarity(query_embedding, item["embedding"]),
        })
    scored.sort(key=lambda row: row["score"], reverse=True)
    return scored[:top_k]


vector_retrieve("I’m facing a ZE-500 error. What are the common root causes behind it, and how can I troubleshoot and resolve the issue effectively?")

Built embeddings for 5 documents.


[{'doc_id': '04_common_errors.txt',
  'title': 'Common Errors and ZE Codes',
  'text': 'Common Errors and ZE Codes\n\nZE-401 is caused by a lane conflict when two patches target the same position. Resolve it with zync lane rebase --origin, which replays your patches on top of the latest remote state.\nZE-302 is caused by an authentication failure, usually from an expired session. Resolve it with zync auth refresh to renew the token without re-entering credentials.\nZE-404 is caused by referencing a lane that does not exist. Resolve with zync lane create <name>, or verify the name with zync lane list.\nZE-500 is usually triggered when a software patch or update becomes corrupted while being downloaded, transferred, or applied. In most cases, this happens because the connection between the client and the update source was interrupted or unstable during the patching process.',
  'score': 0.6717771206821631},
 {'doc_id': '02_design_documentation.txt',
  'title': 'Design Documentation',
  '

## 6. Extract relationships and load Neo4j

Real graph pipelines usually rely on an information extraction layer. That layer might be:

- hand-written rules
- an LLM extraction prompt
- a classical NLP pipeline
- a hybrid validation system

To keep the tutorial transparent, we use **deterministic extraction rules** over the prose. Once the triples are extracted, we load them into Neo4j and retrieve from the graph database directly.


In [ ]:
def add_if_match(triples: list[tuple[str, str, str]], text: str, pattern: str, edges: list[tuple[str, str, str]]):
    """Append fixed triples when a prose pattern is found in a document."""
    if re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL):
        triples.extend(edges)


def extract_triples(document: dict) -> list[tuple[str, str, str]]:
    """Extract a relationship graph from Zync documentation using readable rules."""
    text = document["full_text"]
    triples: list[tuple[str, str, str]] = []

    # Data pipeline flow
    add_if_match(triples, text,
        r"zync upload triggers the Patch Store",
        [("zync upload", "TRIGGERS", "Patch Store")])

    add_if_match(triples, text,
        r"Patch Store validates incoming patches before forwarding them to Lane Tracker",
        [("Patch Store", "FORWARDS_TO", "Lane Tracker")])

    add_if_match(triples, text,
        r"Lane Tracker.*notifies Proposal Engine",
        [("Lane Tracker", "NOTIFIES", "Proposal Engine")])

    add_if_match(triples, text,
        r"Proposal Engine requires Guardian Service",
        [("Proposal Engine", "REQUIRES", "Guardian Service")])

    # Team ownership
    add_if_match(triples, text,
        r"Core Team owns Zync Core",
        [("Core Team", "OWNS", "Zync Core")])

    add_if_match(triples, text,
        r"Infra Team owns the Patch Store",
        [("Infra Team", "OWNS", "Patch Store")])

    add_if_match(triples, text,
        r"Guardian Service is owned by the Core Team",
        [("Core Team", "OWNS", "Guardian Service")])

    # Proposal workflow
    add_if_match(triples, text,
        r"proposal is reviewed by a Guardian",
        [("Proposal", "REVIEWED_BY", "Guardian")])

    add_if_match(triples, text,
        r"Guardians are assigned by the Vault Admin",
        [("Vault Admin", "ASSIGNS", "Guardian")])

    add_if_match(triples, text,
        r"Vault Admin manages Guardian assignments",
        [("Vault Admin", "MANAGES", "Guardian Assignments")])

    add_if_match(triples, text,
        r"Proposal Engine enforces.*Guardian approval",
        [("Proposal Engine", "ENFORCES", "Guardian Approval")])

    # Error codes — causes
    add_if_match(triples, text,
        r"ZE-401 is caused by a lane conflict",
        [("ZE-401", "CAUSED_BY", "Lane Conflict")])

    add_if_match(triples, text,
        r"ZE-302 is caused by an authentication failure",
        [("ZE-302", "CAUSED_BY", "Auth Failure")])

    add_if_match(triples, text,
        r"ZE-404 is caused by referencing a lane that does not exist",
        [("ZE-404", "CAUSED_BY", "Missing Lane")])

    add_if_match(triples, text,
        r"ZE-500 is caused by patch corruption",
        [("ZE-500", "CAUSED_BY", "Patch Corruption")])

    # Error codes — resolutions
    add_if_match(triples, text,
        r"ZE-401.*zync lane rebase|lane conflict.*zync lane rebase",
        [("Lane Conflict", "RESOLVED_BY", "zync lane rebase")])

    add_if_match(triples, text,
        r"ZE-302.*zync auth refresh|authentication failure.*zync auth refresh",
        [("Auth Failure", "RESOLVED_BY", "zync auth refresh")])

    add_if_match(triples, text,
        r"ZE-404.*zync lane create|does not exist.*zync lane create",
        [("Missing Lane", "RESOLVED_BY", "zync lane create")])

    add_if_match(triples, text,
        r"ZE-500.*Escalate.*Vault Admin|patch corruption.*Vault Admin",
        [("Patch Corruption", "ESCALATES_TO", "Vault Admin")])

    add_if_match(triples, text,
        r"Vault Admin.*coordinates with Infra Team.*Patch Store",
        [("Vault Admin", "COORDINATES_WITH", "Infra Team")])

    return list(dict.fromkeys(triples))


def run_cypher(query: str, parameters: dict | None = None, database: str = NEO4J_DATABASE) -> list[dict]:
    """Run a Cypher query and return rows as plain dictionaries."""
    with graph_driver.session(database=database) as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]


relation_rows = []
for document in documents:
    triples = extract_triples(document)
    for source, relation, target in triples:
        relation_rows.append({
            "source": source, "relation": relation, "target": target,
            "doc_id": document["doc_id"], "doc_title": document["title"],
        })

relations_df = pd.DataFrame(relation_rows)

run_cypher("CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (node:Entity) REQUIRE node.name IS UNIQUE")
run_cypher("MATCH (node) DETACH DELETE node")
run_cypher(
    """
    UNWIND $rows AS row
    MERGE (source:Entity {name: row.source})
    MERGE (target:Entity {name: row.target})
    MERGE (source)-[rel:RELATES_TO {relation: row.relation, doc_id: row.doc_id}]->(target)
    SET rel.doc_title = row.doc_title
    """,
    {"rows": relation_rows},
)

graph_counts = run_cypher(
    """
    MATCH (source:Entity)
    WITH count(source) AS node_count
    MATCH ()-[rel:RELATES_TO]->()
    RETURN node_count, count(rel) AS edge_count
    """
)[0]
graph_node_names = [row["name"] for row in run_cypher("MATCH (node:Entity) RETURN node.name AS name ORDER BY name")]

print(f"Neo4j entities: {graph_counts['node_count']} | relationships: {graph_counts['edge_count']}")
relations_df

## 7. Retrieve from the graph database

We use seed-entity matching plus Cypher traversal in Neo4j. Because the graph is small, the retrieval logic stays simple and readable.

In production you would likely add:

- entity normalization
- query decomposition
- path scoring and ranking
- hybrid retrieval fallback


In [ ]:
def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.lower()).strip()


def find_seed_nodes(query: str) -> list[str]:
    """Find graph seed nodes using exact phrase and token-subset matching."""
    query_norm = normalize_text(query)
    query_tokens = set(re.findall(r"[a-zA-Z]+", query_norm))
    seeds = []

    for node in graph_node_names:
        node_norm = normalize_text(node)
        node_tokens = {token for token in re.findall(r"[a-zA-Z]+", node_norm) if len(token) > 2}

        if node_norm in query_norm:
            seeds.append(node)
            continue

        if node_tokens and node_tokens.issubset(query_tokens):
            seeds.append(node)

    return sorted(set(seeds))


def graph_retrieve(query: str, max_hops: int = 2, max_edges: int = 12) -> dict:
    """Retrieve connected facts by traversing Neo4j with Cypher."""
    seed_nodes = find_seed_nodes(query)
    if not seed_nodes:
        return {
            "query": query,
            "seed_nodes": [],
            "edges": [],
            "doc_ids": [],
            "context": "Structured facts:\nNo matching seed entities were found in the graph database.",
        }

    cypher = f"""
    MATCH path = (seed:Entity)-[:RELATES_TO*1..{max_hops}]-(other:Entity)
    WHERE seed.name IN $seed_nodes
    UNWIND relationships(path) AS rel
    WITH DISTINCT rel
    RETURN
        startNode(rel).name AS source,
        rel.relation AS relation,
        endNode(rel).name AS target,
        rel.doc_id AS doc_id,
        rel.doc_title AS doc_title
    LIMIT $max_edges
    """

    unique_edges = run_cypher(cypher, {"seed_nodes": seed_nodes, "max_edges": max_edges})

    supporting_doc_ids = []
    for edge in unique_edges:
        if edge["doc_id"] not in supporting_doc_ids:
            supporting_doc_ids.append(edge["doc_id"])

    doc_lookup = {doc["doc_id"]: doc for doc in documents}
    doc_snippets = []
    for doc_id in supporting_doc_ids:
        snippet = doc_lookup[doc_id]["body"][:350].replace("\n", " ")
        doc_snippets.append(f"[{doc_id}] {snippet}...")

    edge_lines = [
        f"{edge['source']} -> {edge['relation']} -> {edge['target']} (source: {edge['doc_id']})"
        for edge in unique_edges[:max_edges]
    ]

    return {
        "query": query,
        "seed_nodes": seed_nodes,
        "edges": unique_edges[:max_edges],
        "doc_ids": supporting_doc_ids,
        "context": "Structured facts:\n"
        + "\n".join(edge_lines)
        + "\n\nSupporting document snippets:\n"
        + "\n".join(doc_snippets),
    }


graph_retrieve("Which service receives the output when zync upload is run, and which team owns that service?")

## 8. Benchmark questions

These questions all require facts from more than one document to answer completely. This makes them a useful test for comparing both retrieval methods — each method will handle cross-document questions differently depending on how it represents and retrieves knowledge.

In [ ]:
benchmark_questions = [
    {
        "query": "Which service receives the output when zync upload is run, and which team owns that service?",
        "expected_terms": ["patch store", "infra team"],
    },
    {
        "query": "What error code is caused by a lane conflict, and what command resolves it?",
        "expected_terms": ["ze-401", "zync lane rebase"],
    },
    {
        "query": "Who assigns Guardians to a vault, and what service requires Guardian approval before a proposal can be merged?",
        "expected_terms": ["vault admin", "proposal engine"],
    },
    {
        "query": "Which team owns the Patch Store, and which error code requires escalating to the Vault Admin?",
        "expected_terms": ["infra team", "ze-500"],
    },
    {
        "query": "Which service notifies the Proposal Engine, and which team owns Zync Core?",
        "expected_terms": ["lane tracker", "core team"],
    },
    {
        "query": "What causes ZE-302 and what command resolves it?",
        "expected_terms": ["authentication failure", "zync auth refresh"],
    },
]


def supports_expected_terms(text: str, expected_terms: list[str]) -> bool:
    normalized_text = normalize_text(text)
    return all(term.lower() in normalized_text for term in expected_terms)


results = []

for item in benchmark_questions:
    query = item["query"]

    vector_start = time.perf_counter()
    vector_hits = vector_retrieve(query, top_k=1)
    vector_latency_ms = (time.perf_counter() - vector_start) * 1000
    vector_context = "\n\n".join(hit["text"] for hit in vector_hits)

    graph_start = time.perf_counter()
    graph_hits = graph_retrieve(query, max_hops=2, max_edges=12)
    graph_latency_ms = (time.perf_counter() - graph_start) * 1000
    graph_context = graph_hits["context"]

    results.append({
        "query": query,
        "vector_docs": [hit["doc_id"] for hit in vector_hits],
        "graph_docs": graph_hits["doc_ids"],
        "graph_seed_nodes": graph_hits["seed_nodes"],
        "vector_latency_ms": round(vector_latency_ms, 2),
        "graph_latency_ms": round(graph_latency_ms, 2),
        "vector_supports_answer": supports_expected_terms(vector_context, item["expected_terms"]),
        "graph_supports_answer": supports_expected_terms(graph_context, item["expected_terms"]),
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "method": "vector_rag_top1_full_docs",
            "avg_latency_ms": results_df["vector_latency_ms"].mean(),
            "support_rate": results_df["vector_supports_answer"].mean(),
        },
        {
            "method": "graph_rag_neo4j",
            "avg_latency_ms": results_df["graph_latency_ms"].mean(),
            "support_rate": results_df["graph_supports_answer"].mean(),
        },
    ]
).round(3)

summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

latency_df = summary_df.set_index("method")[["avg_latency_ms"]]
latency_df.plot(kind="bar", ax=axes[0], legend=False)
axes[0].set_title("Average retrieval latency")
axes[0].set_xlabel("")
axes[0].set_ylabel("Latency (ms)")

support_df = summary_df.set_index("method")[["support_rate"]]
support_df.plot(kind="bar", ax=axes[1], legend=False)
axes[1].set_title("Answer-support rate")
axes[1].set_xlabel("")
axes[1].set_ylabel("Rate")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 9. Generate answers with both methods

The benchmark above checks whether the retrieved context contains the expected facts. Now we pass that context to the local generation model.

In [ ]:
def build_answer_prompt(method_name: str, query: str, context: str) -> str:
    """Create the same grounding prompt for both retrieval methods."""
    return textwrap.dedent(
        f"""
        You are answering questions about the Zync code collaboration platform.
        Use only the supplied context.
        If the context is incomplete, say so directly.

        Retrieval method: {method_name}
        User question: {query}

        Context:
        {context}

        Answer in 3-5 sentences.
        """
    ).strip()


def answer_with_vector_rag(query: str) -> dict:
    hits = vector_retrieve(query, top_k=1)
    context = "\n\n".join(hit["text"] for hit in hits)
    prompt = build_answer_prompt("vector_rag_top1_full_docs", query, context)
    return {"docs": [hit["doc_id"] for hit in hits], "context": context, "answer": ollama_generate(prompt)}


def answer_with_graph_rag(query: str) -> dict:
    retrieval = graph_retrieve(query, max_hops=2, max_edges=12)
    prompt = build_answer_prompt("graph_rag_neo4j", query, retrieval["context"])
    return {
        "docs": retrieval["doc_ids"],
        "seed_nodes": retrieval["seed_nodes"],
        "context": retrieval["context"],
        "answer": ollama_generate(prompt),
    }


demo_query = "Which service receives the output when zync upload is run, and which team owns that service?"

vector_demo = answer_with_vector_rag(demo_query)
graph_demo  = answer_with_graph_rag(demo_query)

pd.DataFrame([
    {"method": "vector_rag_top1_full_docs", "docs": vector_demo["docs"], "answer": vector_demo["answer"]},
    {"method": "graph_rag_neo4j",           "docs": graph_demo["docs"],  "answer": graph_demo["answer"]},
])

## 10. Interpreting the results

The benchmark questions are designed to require connecting facts across documents. For these questions:

- **Vector retrieval** returns the most topically similar document. Whether it contains the full answer depends on how much of the relevant information is concentrated in the top-ranked document.
- **Graph retrieval** traverses entity relationships, which can pull in relevant facts from multiple documents in a single query — useful when the answer is distributed across the corpus.

For questions that map clearly to a single document, the vector baseline typically performs comparably or better — it is simpler, requires no extraction pipeline, and has lower latency.

The takeaway: neither method is universally better. The question type is the deciding factor.

**Section 13 below** demonstrates five production failure modes that appear in real graph RAG deployments — and shows how to address each one with working code.

## 11. How this maps to production

A real deployment should separate ingestion, extraction, storage, and serving.

```mermaid
flowchart TB
    A[Policies, Playbooks, Notes, Incident Docs] --> B[Ingestion Pipeline]
    B --> C[Chunking + Metadata]
    B --> D[Entity and Relationship Extraction]
    C --> E[Vector Database]
    D --> F[Graph Database]
    G[Query API] --> H[Retriever Router]
    H --> E
    H --> F
    E --> I[Context Builder]
    F --> I
    I --> J[LLM Gateway]
    J --> K[Answer + Observability Trace]
```

### Production upgrades

1. Replace the notebook vector baseline with chunked retrieval in `pgvector`, Qdrant, Weaviate, or Milvus.
2. Move the notebook Neo4j instance behind a managed graph service or a hardened Neo4j deployment.
3. Move extraction into a repeatable ingestion pipeline and validate extracted edges before serving them.
4. Add source attribution, latency monitoring, evaluation harnesses, and access control.
5. Keep the Ollama path for local development, then swap to a production model gateway behind the same interfaces.

A practical architecture often ends up hybrid:

- semantic questions route to vector retrieval
- dependency and ownership questions route to graph retrieval
- high-value workflows use both


## 12. Summary

Both retrieval strategies have a role to play. Vector retrieval is the right starting point for most knowledge bases — it is fast, simple, and works well for single-document lookups and broad semantic questions.

Graph retrieval earns its additional complexity when questions depend on supplier links, team ownership chains, routing fallbacks, or cascading events that span multiple documents. In those cases, traversing explicit relationships tends to surface more of the relevant facts.

A common production architecture is hybrid: vector retrieval for semantic lookup, graph retrieval for relationship traversal, and a router that selects the appropriate path per query type.

See **Section 13** below for five production failure modes and working code fixes.

---

## Section 13 — Production Failure Modes and Fixes

Even a well-designed graph RAG pipeline runs into specific, predictable failure modes when it meets real production workloads. This section demonstrates five of them, shows each failure with live code, and then shows the practical fix.

| # | Failure Mode | Symptom | Fix |
|---|---|---|---|
| 1 | Entity Disambiguation | Seed matching returns zero nodes for name variations | Fuzzy string matching |
| 2 | Brittle Extraction | Paraphrased relationships are silently dropped from the graph | LLM-based extraction |
| 3 | Shallow Graph Depth | Multi-hop answers are incomplete at default hop depth | Adaptive hop depth per query type |
| 4 | Hallucination | LLM answers confidently when context does not support the query | Safe refusal prompt + grounding check |
| 5 | Latency Bottlenecks | P95 latency dominated by repeated embedding model calls | Embedding cache + connection pooling |

### Failure 1 — Entity Disambiguation

**The problem**: Seed node matching compares query tokens against entity names stored in the graph. Real users type entity names with different capitalisation ("patch store" vs "Patch Store"), missing punctuation ("ZE401" vs "ZE-401"), or abbreviated forms ("infra team" vs "Infra Team").

Exact matching returns zero seeds for these queries, so the graph returns empty context — even though the information is in the graph under a slightly different name. The failure is silent: no error, just an empty answer that looks like missing knowledge.

The fix is fuzzy string matching at the seed-finding stage using `SequenceMatcher` or an edit-distance library.

In [ ]:
# Failure 1 Demo: Exact seed matching silently fails on entity name variations

disambiguation_queries = [
    ("What does the Patch store validate?",   "Patch store (lowercase s) vs Patch Store"),
    ("What does the infra team own?",         "infra team (all lowercase) vs Infra Team"),
    ("What is ZE401 caused by?",              "ZE401 (missing dash) vs ZE-401"),
]

print("=== Failure 1: Entity Disambiguation — The Problem ===\n")
for query, note in disambiguation_queries:
    seeds  = find_seed_nodes(query)
    result = graph_retrieve(query)
    print(f"Query     : {query}")
    print(f"Note      : {note}")
    print(f"Seed nodes: {seeds or '(none)'} — {'MISS: empty graph context' if not seeds else 'OK'}")
    print()

print("All three queries refer to entities that exist in the graph.")
print("Exact matching returns zero seeds — the system silently falls back to empty context.")

In [ ]:
# Failure 1 Fix: Fuzzy seed node matching handles name variations

from difflib import SequenceMatcher

def fuzzy_find_seed_nodes(query: str, threshold: float = 0.82) -> list[str]:
    """Fuzzy seed node matching — handles abbreviations, singular/plural, and minor typos."""
    query_norm = normalize_text(query)
    query_tokens = set(re.findall(r"[a-zA-Z]+", query_norm))
    seeds = []

    for node in graph_node_names:
        node_norm = normalize_text(node)
        node_tokens = {t for t in re.findall(r"[a-zA-Z]+", node_norm) if len(t) > 2}

        if node_norm in query_norm:
            seeds.append(node); continue
        if node_tokens and node_tokens.issubset(query_tokens):
            seeds.append(node); continue

        # Sliding-window fuzzy match over query words
        node_words  = node_norm.split()
        query_words = query_norm.split()
        for i in range(max(1, len(query_words) - len(node_words) + 1)):
            window = " ".join(query_words[i : i + len(node_words)])
            if SequenceMatcher(None, node_norm, window).ratio() >= threshold:
                seeds.append(node)
                break

    return sorted(set(seeds))


print("=== Failure 1 Fix: Fuzzy Matching ===\n")
for query, note in disambiguation_queries:
    exact     = find_seed_nodes(query)
    fuzzy     = fuzzy_find_seed_nodes(query)
    recovered = [s for s in fuzzy if s not in exact]
    print(f"Query     : {query}")
    print(f"Exact     : {exact or '(none)'}")
    print(f"Fuzzy     : {fuzzy or '(none)'}")
    if recovered:
        print(f"Recovered : {recovered}")
    print()

print("Production tip: tune the threshold per domain (0.80-0.88 is typical).")
print("For multi-word entities, also use embedding similarity as a second-pass fallback.")

### Failure 2 — Brittle Extraction Patterns

**The problem**: Rule-based (regex) extraction only matches the exact phrasings it was written for. When a new document uses a paraphrase — *"The central storage layer checks each incoming change before routing it to the lane management service"* instead of *"Patch Store validates incoming patches before forwarding them to Lane Tracker"* — the pattern misses the relationship entirely.

The graph quietly becomes incomplete as new documents arrive with different vocabulary. Queries over missing relationships silently return partial results without any error.

The fix is LLM-based extraction, which understands meaning rather than matching surface text.

In [ ]:
# Failure 2 Demo: Regex extraction returns zero triples for a paraphrased document

paraphrased_doc = {
    "doc_id": "paraphrase_test.txt",
    "title": "Paraphrase Test",
    "full_text": (
        "Paraphrase Test\n\n"
        "The central storage layer checks each incoming change before routing it to the lane "
        "management service. The merge workflow needs sign-off from the approval service before "
        "changes can land in a protected lane. "
        "The platform engineering group is accountable for the stability of the core routing "
        "and storage components."
    ),
}

original_doc_snippet = {
    "doc_id": "original_test.txt",
    "title": "Original Phrasing",
    "full_text": (
        "Original Phrasing\n\n"
        "Patch Store validates incoming patches before forwarding them to Lane Tracker. "
        "Proposal Engine requires Guardian Service approval before a proposal can be merged into a protected lane. "
        "Core Team owns Zync Core."
    ),
}

orig_triples = extract_triples(original_doc_snippet)
para_triples = extract_triples(paraphrased_doc)

print("=== Failure 2: Brittle Extraction — The Problem ===\n")
print(f"Original phrasing   → {len(orig_triples)} triples extracted:")
for t in orig_triples:
    print(f"  {t}")

print(f"\nParaphrased phrasing → {len(para_triples)} triples extracted:")
if para_triples:
    for t in para_triples:
        print(f"  {t}")
else:
    print("  (none — all patterns missed the paraphrase)")
print()
print("Same relationships, different words. The regex extractor is blind to the paraphrase.")

In [ ]:
# Failure 2 Fix: LLM-based extraction handles paraphrasing

EXTRACTION_PROMPT = '''Extract entity relationships from the text below as (subject | relation | object) triples.

Focus on:
- Supply and dependency (supplies, requires, depends on, provides)
- Ownership and responsibility (owns, manages, is accountable for)
- Data flow (sends data to, forwards to, flows through)
- Risk and disruption (disrupts, delays, impacts)
- Routing and fallback (routes through, backs up, protects)

Return ONLY triples in this exact format, one per line:
subject | relation | object

Do not include any explanation. Only output triples.

Text:
{text}

Triples:'''


def extract_triples_with_llm(document: dict) -> list[tuple[str, str, str]]:
    """LLM-based extraction — handles paraphrasing that regex patterns miss."""
    prompt = EXTRACTION_PROMPT.format(text=document["full_text"][:2000])
    raw = ollama_generate(prompt)
    triples = []
    for line in raw.strip().splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) == 3 and all(p for p in parts):
            triples.append((parts[0], parts[1], parts[2]))
    return triples


print("=== Failure 2 Fix: LLM-Based Extraction ===\n")
print("Running LLM extraction on the paraphrased document...\n")
llm_triples = extract_triples_with_llm(paraphrased_doc)

if llm_triples:
    print(f"LLM extracted {len(llm_triples)} triples:")
    for t in llm_triples:
        print(f"  {t[0]} | {t[1]} | {t[2]}")
else:
    print("No triples returned (check that the generation model is running).")

print()
print("Production note: validate LLM-extracted triples before ingesting into the graph.")
print("A practical quality gate: spot-check 5-10% of triples per batch and reject any")
print("triple where the extracted relation contradicts the source sentence.")

### Failure 3 — Shallow Graph Depth (Missing Multi-Hop Answers)

**The problem**: The graph contains all the facts, but the retrieval does not traverse deep enough to connect them.

Consider: *"If a developer runs zync upload and gets ZE-500, who should they escalate to?"*

The answer requires a 4-hop chain:
```
zync upload → TRIGGERS → Patch Store
Patch Store failure → emits → ZE-500
ZE-500 → CAUSED_BY → Patch Corruption
Patch Corruption → ESCALATES_TO → Vault Admin
```

With `max_hops=1` you see only the first link. The answer only fully assembles at `max_hops=3`. A fixed low value silently returns an incomplete answer with no error or warning.

In [ ]:
# Failure 3 Demo: Multi-hop question with increasing hop depth

three_hop_query = "If a developer runs zync upload and gets ZE-500, who should they escalate to?"

print("=== Failure 3: Shallow Hops — The Problem ===\n")
print(f"Query: {three_hop_query}\n")
print("Required reasoning chain:")
print("  zync upload  --TRIGGERS-->  Patch Store")
print("  Patch Store failure  -->  ZE-500 emitted")
print("  ZE-500  --CAUSED_BY-->  Patch Corruption")
print("  Patch Corruption  --ESCALATES_TO-->  Vault Admin\n")

chain_check = ["zync upload", "patch store", "ze-500", "vault admin"]

for hops in [1, 2, 3]:
    result = graph_retrieve(three_hop_query, max_hops=hops, max_edges=30)
    ctx = result["context"].lower()
    covered = [e for e in chain_check if e in ctx]
    missing  = [e for e in chain_check if e not in ctx]
    print(f"max_hops={hops}: {len(result['edges'])} edges | seeds={result['seed_nodes']}")
    print(f"  Chain covered : {covered}")
    print(f"  Still missing : {missing}")
    print()

print("Production practice: classify queries by depth requirement, not one-size-fits-all.")
print("  Lookup (what does ZE-401 mean?)              → max_hops=1")
print("  Dependency (what triggers Patch Store?)      → max_hops=2")
print("  Cascading impact (zync upload → who owns?) → max_hops=3")

### Failure 4 — Hallucination and Grounding Failures

**The problem**: LLMs are trained to produce helpful, fluent responses. When retrieved context is topically close but does not actually contain the answer, the model tends to fill the gap with plausible-sounding content rather than admitting uncertainty.

This is especially common in vector RAG when the top document has high cosine similarity on surface words ("Zync", "vault") but does not contain the specific fact requested ("who founded it", "when was it released").

Two mitigations work together:
- A **safe refusal prompt** that explicitly instructs the model to say "I don't know" when context is insufficient.
- A **post-hoc grounding check** that asks the same LLM to verify whether its answer is supported by the context.

In [ ]:
# Failure 4 Demo: Vector RAG retrieves a plausible-sounding but non-answering document

off_domain_query = "Who founded the Zync platform and when was it first released publicly?"

vector_hits = vector_retrieve(off_domain_query, top_k=1)
vector_context = vector_hits[0]["text"]

print("=== Failure 4: Hallucination — The Problem ===\n")
print(f"Query    : {off_domain_query}")
print(f"Retrieved: {vector_hits[0]['doc_id']} (similarity={vector_hits[0]['score']:.3f})")
print()
print("The retrieved document is about Zync, but contains no founding or release history.")
print("Passing this context to the LLM anyway...\n")

hallucinated_answer = ollama_generate(build_answer_prompt("vector_rag", off_domain_query, vector_context))
print(f"LLM Answer:\n{hallucinated_answer}")
print()
print("The LLM may have fabricated founding dates or names not found anywhere in the corpus.")

In [ ]:
# Failure 4 Fix: Safe prompt + post-hoc grounding check

SAFE_PROMPT = '''You are a Zync platform knowledge assistant.
Answer using ONLY the context provided. If the context does not contain enough
information to answer confidently, respond with exactly:
"I don't have enough information in the current knowledge base to answer that."

Context:
{context}

Question: {query}

Answer:'''

GROUNDING_CHECK_PROMPT = '''Does the answer below rely only on facts explicitly present in the context?

Context (first 600 chars):
{context}

Answer:
{answer}

Reply with GROUNDED or UNGROUNDED, then one sentence explaining your decision.

Verdict:'''


def safe_vector_rag_answer(query: str) -> dict:
    """Vector retrieval with a safe refusal prompt and post-hoc grounding check."""
    hits = vector_retrieve(query, top_k=1)
    context = hits[0]["text"] if hits else "No documents retrieved."
    safe_answer = ollama_generate(SAFE_PROMPT.format(context=context[:2000], query=query))
    grounding_raw = ollama_generate(
        GROUNDING_CHECK_PROMPT.format(context=context[:600], answer=safe_answer)
    )
    verdict = "GROUNDED" if grounding_raw.strip().upper().startswith("GROUNDED") else "UNGROUNDED"
    return {
        "docs": [h["doc_id"] for h in hits],
        "answer": safe_answer,
        "grounding_verdict": verdict,
        "grounding_explanation": grounding_raw,
    }


print("=== Failure 4 Fix: Safe Prompt + Grounding Check ===\n")
safe_result = safe_vector_rag_answer(off_domain_query)
print(f"Answer:\n{safe_result['answer']}")
print(f"\nGrounding verdict: {safe_result['grounding_verdict']}")
print(f"Explanation: {safe_result['grounding_explanation']}")
print()
print("Production notes:")
print("  - Log all UNGROUNDED verdicts for human review.")
print("  - If UNGROUNDED rate > 5%, retrieval is returning the wrong documents.")
print("  - Hard rule: if max similarity score < 0.50, skip the LLM and return 'insufficient context'.")

### Failure 5 — Latency Bottlenecks

**The problem**: Two bottlenecks dominate RAG query latency in practice:

1. **Embedding computation** — every query hits the embedding model. Locally with Ollama this is ~100–600ms per call. Repeated identical queries waste all of that time.
2. **Cold graph traversal** — the first Cypher query per Neo4j session triggers query planning; subsequent queries on the same session are faster.

Both are fixable: an embedding cache eliminates redundant model calls, and connection pooling keeps Neo4j sessions warm across requests.

In [ ]:
# Failure 5: Measure embedding and graph traversal latency, then apply caching

print("=== Failure 5: Latency Bottlenecks ===\n")

latency_query = "Which service receives the output when zync upload is run?"

# Graph traversal latency (5 runs)
print("Graph traversal latency (5 runs):")
graph_times = []
for _ in range(5):
    t0 = time.perf_counter()
    graph_retrieve(latency_query)
    graph_times.append((time.perf_counter() - t0) * 1000)
print(f"  Run times (ms): {[round(t, 1) for t in graph_times]}")
print(f"  Mean: {sum(graph_times)/len(graph_times):.1f}ms  Min: {min(graph_times):.1f}ms\n")

# Embedding latency — uncached
print("Embedding latency — uncached (3 runs):")
uncached_times = []
for _ in range(3):
    t0 = time.perf_counter()
    ollama_embed(latency_query)
    uncached_times.append((time.perf_counter() - t0) * 1000)
print(f"  {[round(t, 1) for t in uncached_times]}ms\n")

# Fix: embedding cache
_embed_cache: dict[str, np.ndarray] = {}

def cached_embed(text: str) -> np.ndarray:
    """Cache embeddings by normalized content to skip recomputing identical queries."""
    key = normalize_text(text)[:300]
    if key not in _embed_cache:
        _embed_cache[key] = ollama_embed(text)
    return _embed_cache[key]

_ = cached_embed(latency_query)  # warm the cache

print("Embedding latency — cached (3 runs, after warm-up):")
cached_times = []
for _ in range(3):
    t0 = time.perf_counter()
    cached_embed(latency_query)
    cached_times.append((time.perf_counter() - t0) * 1000)
print(f"  {[round(t, 1) for t in cached_times]}ms")
speedup = sum(uncached_times) / max(sum(cached_times), 0.01)
print(f"\nSpeedup for repeated queries: ~{speedup:.0f}x\n")

print("Production notes:")
print("  - Use Redis or an in-process LRU cache; set TTL to match document update frequency.")
print("  - For graph queries: use Neo4j connection pooling, keep sessions warm across requests.")
print("  - For SLAs < 200ms: pre-embed a set of high-frequency query phrases at service startup.")

### Production Lessons Summary

| Failure | Symptom | Fix |
|---|---|---|
| Entity Disambiguation | Silent empty context for queries using name variations | Fuzzy matching (SequenceMatcher, or embedding-based entity lookup) |
| Brittle Extraction | New documents add zero graph edges; paraphrased relationships invisible | LLM-based extraction with batch validation sampling |
| Shallow Hops | Multi-step answers return partial chains; final link always missing | Query classification → adaptive `max_hops` per question type |
| Hallucination | Confident wrong answers when context does not support the query | Safe refusal prompt + post-hoc grounding check + similarity threshold floor |
| Latency Bottlenecks | P95 latency dominated by repeated embedding model calls | Embedding cache (Redis/LRU) + Neo4j connection pooling |

**When graph RAG is worth the investment**

Graph RAG earns its operational cost when your question corpus includes:
- *Dependency questions*: "What does X depend on?" or "What blocks Y?"
- *Ownership questions*: "Who is responsible for X when Y fails?"
- *Cascading impact questions*: "If X is disrupted, what is affected downstream?"
- *Multi-document synthesis*: "Combine the supply chain policy, the incident retro, and the launch brief to answer..."

For simpler lookup questions ("What does the handbook say about travel reimbursement?"), a well-chunked vector retriever with metadata filters is usually sufficient and far simpler to operate.